### Prepare BED for [--mane] in mapping module
- MANE Select/MANE Plus Clinical transcript + junction

In [ ]:
import gzip
from collections import defaultdict

'''
BED of MANE_Select and MANE_Plus_Clinical (5'UTR, [CDS ... junction], 3'UTR)

# fetch the gencode basic annotation
wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.basic.annotation.gff3.gz

'''

GFF = "./content/gencode.v49.basic.annotation.gff3.gz"
OUT = "./content/MANE.gencode_v49.annotation.bed.gz"


KEEP_TAGS = {
    "MANE_Select",
    "MANE_Plus_Clinical"
}

JUNC_SIZE = 20


def parse_attrs(attr):
    """
    Parse GFF3 attributes
    """
    d = {}
    for x in attr.split(";"):
        if "=" in x:
            k, v = x.split("=", 1)
            d[k] = v
    return d

# -------------------------
# read GFF
# -------------------------

transcripts = {}

features = defaultdict(list)


with gzip.open(GFF, "rt") as f:

    for line in f:

        if line.startswith("#"):
            continue

        chrom, source, feature, start, end, score, strand, phase, attr = \
            line.rstrip().split("\t")

        attrs = parse_attrs(attr)

        # MANE transcript
        tags = attrs.get("tag", "").split(",")
        mane_tag = next(
            (tag for tag in tags if tag in KEEP_TAGS),
            None
        )

        if mane_tag is None:
            continue

        # CDS and UTR
        if feature in ["exon", "CDS", "three_prime_UTR", "five_prime_UTR"]:

            gene_name = attrs.get("gene_name")
            tid = attrs.get("transcript_id")
            exon_number = attrs.get("exon_number")

            key = (tid, exon_number, feature)

            transcripts[key] = {
                "chrom"       : str(chrom),
                "start"       : int(start),
                "end"         : int(end),
                "tid"         : str(tid),
                "strand"      : str(strand),
                "feature"     : str(feature),
                "exon_number" : str(exon_number),
                "gene_name"   : str(gene_name),
                "mane_tag"    : str(mane_tag),
            }

# -------------------------
# junction

# gene on + strand:
# exon1 -----> intron -----> exon2
#      donor             acceptor

# gene on - strand:
# exon2 <----- intron <----- exon1
#      acceptor          donor
# -------------------------

# group exons by transcript
exons = defaultdict(list)

for rec in transcripts.values():
    if rec["feature"] == "exon":
        exons[rec["tid"]].append(rec)

junctions = []

for tid, exon_list in exons.items():

    exon_list = sorted(
        exon_list,
        key=lambda x: x["start"]
    )

    strand = exon_list[0]["strand"]

    for i in range(len(exon_list)-1):

        e1 = exon_list[i]
        e2 = exon_list[i+1]

        if strand == "+":

            # donor: end of exon -> intron
            junctions.append({
                "chrom"       : e1["chrom"],
                "start"       : e1["end"],
                "end"         : e1["end"] + JUNC_SIZE,
                "tid"         : tid,
                "strand"      : strand,
                "feature"     : "junction_donor",
                "gene_name"   : e1["gene_name"],
                "exon_number" : e1["exon_number"],
                "mane_tag"    : e1["mane_tag"],
            })

            # acceptor: intron -> start of exon
            junctions.append({
                "chrom"       : e2["chrom"],
                "start"       : e2["start"] - JUNC_SIZE,
                "end"         : e2["start"],
                "tid"         : tid,
                "strand"      : strand,
                "feature"     : "junction_acceptor",
                "gene_name"   : e2["gene_name"],
                "exon_number" : e2["exon_number"],
                "mane_tag"    : e2["mane_tag"],
            })

        else:
            # negative strand
            # donor: exon start -> intron
            junctions.append({
                "chrom"       : e2["chrom"],
                "start"       : e2["start"],
                "end"         : e2["start"] + JUNC_SIZE,
                "tid"         : tid,
                "strand"      : strand,
                "feature"     : "junction_donor",
                "gene_name"   : e2["gene_name"],
                "exon_number" : e2["exon_number"],
                "mane_tag"    : e2["mane_tag"],
            })

            # acceptor: intron -> exon end
            junctions.append({
                "chrom"       : e1["chrom"],
                "start"       : e1["end"] - JUNC_SIZE,
                "end"         : e1["end"],
                "tid"         : tid,
                "strand"      : strand,
                "feature"     : "junction_acceptor",
                "gene_name"   : e1["gene_name"],
                "exon_number" : e1["exon_number"],
                "mane_tag"    : e1["mane_tag"],
            })

# -------------------------
# export
# -------------------------

records = list(transcripts.values()) + junctions

with gzip.open(OUT, "wt") as out:

    for rec in sorted(
        records,
        key=lambda x: (
            x["chrom"],
            x["start"],
            x["end"],
            x["tid"],
            x["strand"],
            x["feature"],
            x["exon_number"],
        ),
    ):

        if rec["feature"] == "exon":
            continue

        # BED: 0-based start, end is unchanged
        bed_start = rec["start"] - 1
        bed_end   = rec["end"]

        out.write(
            "\t".join(
                [
                    rec["chrom"],
                    # join need string
                    str(bed_start),
                    str(bed_end),
                    rec["tid"],
                    rec["strand"],
                    rec["feature"],
                    rec["gene_name"],
                    rec["exon_number"],
                    rec["mane_tag"],
                ]
            )
            + "\n"
            )

### Prepare gnomAD control MEIs for [--control-dir] in mapping module

In [ ]:
###
# gnomad control MEIs (Shell script)

# OUTPUT :
# gnomad.v4.1.sv.sites.INS.{LINE1,ALU,SVA,HERVK}.PASS.vcf.gz
# gnomad.v4.1.sv.sites.DEL.{LINE1,ALU,SVA,HERVK}.PASS.vcf.gz
###

wget https://storage.googleapis.com/gcp-public-data--gnomad/release/4.1/genome_sv/gnomad.v4.1.sv.sites.vcf.gz

bcftools="/Everythings/misc/bcftools-1.21"

for ME in LINE1 ALU SVA HERVK; do
    ${bcftools}/bcftools view \
        -i "ALT=\"<DEL:ME:${ME}>\" && FILTER=\"PASS\"" \
        -Oz --threads 16 \
        -o gnomad.v4.1.sv.sites.DEL.${ME}.PASS.vcf.gz \
        gnomad.v4.1.sv.sites.vcf.gz

done

for ME in LINE1 ALU SVA HERVK; do
    ${bcftools}/bcftools view \
        -i "ALT=\"<INS:ME:${ME}>\" && FILTER=\"PASS\"" \
        -Oz --threads 16 \
        -o gnomad.v4.1.sv.sites.INS.${ME}.PASS.vcf.gz \
        gnomad.v4.1.sv.sites.vcf.gz

done